# 🎬 Zoom → YouTube 自動転送ツール

---

## このツールでできること

Zoomのクラウド録画を、指定した期間分まとめてYouTubeへ自動でアップロードします。
面倒なダウンロード・再アップロード作業をすべて自動化します。

---

## よくある質問（FAQ）

### 💰 費用はかかりますか？
**完全無料です。** Google Colab・Zoom・YouTubeの無料枠の範囲内で動作します。

### 💾 Googleドライブの容量を使いますか？
**使いません。** 動画はYouTubeへのアップロードが完了した直後に自動削除されます。
Colab上の一時領域だけを使うため、Googleドライブの15GB制限には影響しません。

### 🔒 入力したAPIキーは安全ですか？
**安全です。** 入力した情報はあなた自身のGoogleアカウント内のみで使用されます。
このノートブックや外部サービスへ送信されることは一切ありません。

### ⏱ 長い動画をアップロードできますか？
**15分以上の動画をアップロードするには、YouTube側で電話番号認証が必要です。**
YouTubeの設定（youtube.com/verify）から事前に認証しておいてください。

### 📊 1日に何件アップロードできますか？
YouTubeのAPIには1日あたりの上限（約6件）があります。
上限に達した場合は翌日に続きから実行してください。

---

## 使い方の流れ

1. **セクション②** のガイドに従って、ZoomとYouTubeのAPI設定を済ませる
2. **セクション③** のセットアップセルを実行する
3. **セクション④** で転送したい期間や公開設定を入力する
4. **セクション⑤** の実行ボタンを押すと自動転送が始まる


---

## ② API設定ガイド

初回のみ必要な設定です。一度済ませれば次回以降は不要です。

---

### STEP 1：Zoom API の設定

1. [Zoom App Marketplace](https://marketplace.zoom.us/) にアクセスし、Zoomアカウントでログインします
2. 右上の「**Develop**」→「**Build App**」をクリックします
3. 「**Server-to-Server OAuth**」を選択して「**Create**」をクリックします
4. アプリ名を自由に入力して（例：`MyZoomTransfer`）作成します
5. 「**App Credentials**」のタブを開き、以下の3つをメモします
   - **Account ID**
   - **Client ID**
   - **Client Secret**
6. 「**Scopes**」タブを開き、以下の権限を追加します
   - `recording:read:admin`（クラウド録画の読み取り）
7. 「**Activation**」タブでアプリを有効化します

> ⚠️ **Colabシークレットへの保存（重要）**
> 左メニューの 🔑 鍵アイコンを開き、以下の3つを追加してください：
> - `ZOOM_ACCOUNT_ID`（値：メモしたAccount ID）
> - `ZOOM_CLIENT_ID`（値：メモしたClient ID）
> - `ZOOM_CLIENT_SECRET`（値：メモしたClient Secret）

---

### STEP 2：YouTube API の設定

1. [Google Cloud Console](https://console.cloud.google.com/) にアクセスします
2. 画面上部のプロジェクト選択から「**新しいプロジェクト**」を作成します（例：`ZoomYouTubeTransfer`）
3. 左メニューの「**APIとサービス**」→「**ライブラリ**」を開きます
4. 「**YouTube Data API v3**」を検索して「**有効にする**」をクリックします
5. 「**APIとサービス**」→「**認証情報**」を開きます
6. 「**認証情報を作成**」→「**OAuthクライアントID**」を選択します
7. 「**デスクトップアプリ**」を選び、名前を入力して「**作成**」をクリックします
8. 作成されたクライアントIDの右にある ⬇ ダウンロードアイコンを押し、
   `client_secrets.json` をパソコンに保存します

> 💡 **OAuthの同意画面について**
> 「OAuth同意画面」の設定を求められた場合は、「外部」を選んで必要項目（アプリ名・メールアドレス）を入力してください。
> 「テストユーザー」に自分のGoogleアカウントを追加しておくと認証がスムーズです。

---

### STEP 3：実行の準備

上記の設定が済んだら、以下のセルを**上から順番に**実行してください。
各セルの左にある ▶ ボタンをクリックすると実行できます。


In [ ]:
#@title ③ セットアップ（最初に一度だけ実行してください）

import subprocess, sys

print("必要なライブラリをインストールしています。少々お待ちください...")

packages = [
    "google-api-python-client",
    "google-auth-oauthlib",
    "requests",
    "tqdm",
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"⚠️  {pkg} のインストール中に問題が発生しました: {result.stderr}")

print()
print("✅ 準備が整いました！次のセルへ進んでください。")


In [ ]:
#@title ④ 転送設定

#@markdown ### 📅 取得する録画の期間
#@markdown 日付フィールドの右端にある 📅 アイコンをクリックするとカレンダーで選択できます。
start_date = "2024-01-01" #@param {type:"date"}
end_date = "2024-12-31" #@param {type:"date"}

#@markdown ---
#@markdown ### 🎬 YouTubeの公開設定
privacy_status = "private" #@param ["public", "private", "unlisted"]
#@markdown - **private**（非公開）：自分だけが視聴できます（おすすめ）
#@markdown - **unlisted**（限定公開）：URLを知っている人だけが視聴できます
#@markdown - **public**（公開）：誰でも視聴できます

#@markdown ---
#@markdown ### 📝 動画タイトルの接頭辞
title_prefix = "[アーカイブ]" #@param {type:"string"}
#@markdown 動画タイトルの先頭に付けるテキストです。不要な場合は空欄にしてください。

print(f"✅ 設定完了！")
print(f"  期間     : {start_date} ～ {end_date}")
print(f"  公開設定 : {privacy_status}")
print(f"  接頭辞   : '{title_prefix}'")
print()
print("次のセル（⑤ YouTube認証）へ進んでください。")


In [ ]:
#@title ⑤ YouTube認証

import os
import pickle
from google.colab import files
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

# http://localhost へのリダイレクトを許可（Colab環境では必須）
os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

YOUTUBE_SCOPES = ["https://www.googleapis.com/auth/youtube.upload"]
TOKEN_PICKLE_PATH = "/tmp/youtube_oauth_token.pkl"

# 前回の認証情報が残っていれば再利用する
creds = None
if os.path.exists(TOKEN_PICKLE_PATH):
    with open(TOKEN_PICKLE_PATH, "rb") as f:
        creds = pickle.load(f)

if creds and creds.valid:
    print("✅ 前回の認証情報を再利用します。次のセル（⑥ 転送実行）へ進んでください。")
elif creds and creds.expired and creds.refresh_token:
    print("🔄 認証トークンを更新中...")
    creds.refresh(Request())
    with open(TOKEN_PICKLE_PATH, "wb") as f:
        pickle.dump(creds, f)
    print("✅ 更新完了。次のセル（⑥ 転送実行）へ進んでください。")
else:
    # 新規認証
    print("📁 Google Cloud Console からダウンロードした client_secrets.json をアップロードしてください。")
    uploaded = files.upload()
    if not uploaded:
        print("❌ ファイルがアップロードされませんでした。このセルをもう一度実行してください。")
        raise SystemExit(1)
    client_secrets_path = list(uploaded.keys())[0]
    print(f"✅ 認証ファイルを受け取りました: {client_secrets_path}")

    flow = InstalledAppFlow.from_client_secrets_file(
        client_secrets_path,
        scopes=YOUTUBE_SCOPES,
        redirect_uri="http://localhost",
    )
    auth_url, _ = flow.authorization_url(prompt="consent", access_type="offline")

    print()
    print("=" * 60)
    print("【YouTube認証が必要です】以下の手順で進めてください")
    print("=" * 60)
    print()
    print("① 下のURLをコピーしてブラウザで開く")
    print()
    print(auth_url)
    print()
    print("② Googleアカウントにログインして「許可」をクリック")
    print()
    print("③ ブラウザに「このサイトにアクセスできません」と表示される")
    print("   ┗ ✅ これは正常です。エラーではありません。このまま進んでください。")
    print()
    print("④ そのページのアドレスバーに表示されている")
    print("   「http://localhost/?code=...」から始まる長いURLを")
    print("   まるごとコピーして、下の入力欄に貼り付けてEnterを押す")
    print()
    print("=" * 60)

    redirect_response = input("\nリダイレクト後のURL: ").strip()

    try:
        flow.fetch_token(authorization_response=redirect_response)
        creds = flow.credentials
        with open(TOKEN_PICKLE_PATH, "wb") as f:
            pickle.dump(creds, f)
        print()
        print("✅ YouTube認証が完了しました！次のセル（⑥ 転送実行）へ進んでください。")
    except Exception as e:
        print()
        print(f"❌ 認証に失敗しました: {e}")
        print()
        print("よくある原因：")
        print("  ・URLが途中で切れている → アドレスバーのURLを最初から最後までコピーしてください")
        print("  ・URLをページの内容からコピーした → アドレスバー（上部の入力欄）からコピーしてください")
        print()
        print("このセルをもう一度実行してやり直してください。")
        raise SystemExit(1)

youtube = build("youtube", "v3", credentials=creds)


In [ ]:
#@title ⑥ 転送実行

import os
import requests
from tqdm import tqdm
from google.colab import userdata
from googleapiclient.http import MediaFileUpload

os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

# ④ のウィジェットから設定値を読み取る
try:
    start_date = start_picker.value.strftime("%Y-%m-%d")
    end_date = end_picker.value.strftime("%Y-%m-%d")
    privacy_status = privacy_dropdown.value
    title_prefix = prefix_input.value.strip()
except NameError:
    print("❌ エラー：④ 転送設定のセルを先に実行してください。")
    raise SystemExit(1)

print(f"設定確認：{start_date} ～ {end_date}　公開設定：{privacy_status}　接頭辞：'{title_prefix}'")
print()


# ================================================================
# Zoom 関連の関数
# ================================================================

def get_zoom_access_token(account_id, client_id, client_secret):
    url = "https://zoom.us/oauth/token"
    params = {"grant_type": "account_credentials", "account_id": account_id}
    response = requests.post(url, params=params, auth=(client_id, client_secret))
    if response.status_code != 200:
        raise Exception(f"Zoom認証失敗 (ステータス {response.status_code}): {response.text}")
    return response.json()["access_token"]


def get_zoom_recordings(token, from_date, to_date):
    headers = {"Authorization": f"Bearer {token}"}
    recordings = []
    params = {"from": from_date, "to": to_date, "page_size": 300}
    url = "https://api.zoom.us/v2/users/me/recordings"

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"録画リスト取得失敗 (ステータス {response.status_code}): {response.text}")

        data = response.json()
        for meeting in data.get("meetings", []):
            for rec_file in meeting.get("recording_files", []):
                if rec_file.get("file_type") == "MP4" and rec_file.get("status") == "completed":
                    recordings.append({
                        "topic": meeting.get("topic", "無題"),
                        "start_time": meeting.get("start_time", ""),
                        "download_url": rec_file.get("download_url", ""),
                        "file_size": rec_file.get("file_size", 0),
                    })

        next_page_token = data.get("next_page_token", "")
        if not next_page_token:
            break
        params["next_page_token"] = next_page_token

    return recordings


def download_recording(download_url, token, save_path):
    url_with_token = f"{download_url}?access_token={token}"
    response = requests.get(url_with_token, stream=True)
    if response.status_code != 200:
        raise Exception(f"ダウンロード失敗 (ステータス {response.status_code})")

    total_size = int(response.headers.get("content-length", 0))
    with open(save_path, "wb") as f, tqdm(
        desc="  ダウンロード中",
        total=total_size,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        leave=False,
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))


def upload_to_youtube(youtube, filepath, title, description, privacy_status):
    file_size = os.path.getsize(filepath)
    body = {
        "snippet": {
            "title": title[:100],
            "description": description,
            "categoryId": "22",
        },
        "status": {
            "privacyStatus": privacy_status,
            "selfDeclaredMadeForKids": False,
        },
    }
    media = MediaFileUpload(filepath, mimetype="video/mp4", resumable=True, chunksize=5 * 1024 * 1024)
    request = youtube.videos().insert(part="snippet,status", body=body, media_body=media)

    video_id = None
    with tqdm(total=file_size, unit="B", unit_scale=True, unit_divisor=1024, desc="  アップロード中", leave=False) as bar:
        last_progress = 0
        while video_id is None:
            status, response = request.next_chunk()
            if status:
                current = int(status.resumable_progress)
                bar.update(current - last_progress)
                last_progress = current
            if response is not None:
                video_id = response.get("id")
                bar.update(file_size - last_progress)
    return video_id


def japanese_error_hint(error_str):
    e = error_str.lower()
    if "quotaexceeded" in e or "quota" in e:
        return "YouTubeのAPIの1日あたりの上限に達した可能性があります。明日再度お試しください。"
    elif "forbidden" in e or "403" in e:
        return "アクセスが拒否されました。YouTubeの電話番号認証が済んでいるか確認してください。"
    elif "unauthorized" in e or "401" in e:
        return "認証が無効です。⑤ YouTube認証のセルをもう一度実行してください。"
    elif "download" in e:
        return "Zoomからのダウンロードに失敗しました。Zoomの設定を確認してください。"
    elif "connectionerror" in e or "timeout" in e:
        return "ネットワークエラーです。接続を確認して再度お試しください。"
    else:
        return "設定を確認してください。問題が続く場合はエラーメッセージをお知らせください。"


# ================================================================
# メイン処理
# ================================================================

print("=" * 60)
print(" Zoom → YouTube 自動転送ツール")
print("=" * 60)
print()

# ---- YouTube接続確認 ----
print("[1/3] YouTube接続を確認中...")
try:
    _ = youtube
    print("  ✅ 完了")
except NameError:
    print("  ❌ エラー：YouTube認証が完了していません。")
    print("  ⑤ YouTube認証のセルを先に実行してください。")
    raise SystemExit(1)

# ---- Zoom接続 ----
print()
print("[2/3] Zoomへ接続中...")
try:
    zoom_account_id = userdata.get("ZOOM_ACCOUNT_ID")
    zoom_client_id = userdata.get("ZOOM_CLIENT_ID")
    zoom_client_secret = userdata.get("ZOOM_CLIENT_SECRET")
    zoom_token = get_zoom_access_token(zoom_account_id, zoom_client_id, zoom_client_secret)
    print("  ✅ 接続成功")
except Exception as e:
    print(f"  ❌ Zoom接続エラー: {e}")
    print(f"  ⚠️  {japanese_error_hint(str(e))}")
    raise SystemExit(1)

# ---- 録画リスト取得 ----
print()
print(f"[3/3] {start_date} ～ {end_date} の録画を検索中...")
try:
    recordings = get_zoom_recordings(zoom_token, start_date, end_date)
except Exception as e:
    print(f"  ❌ 録画リスト取得エラー: {e}")
    print("  ⚠️  日付の形式が YYYY-MM-DD になっているか確認してください。")
    print("      またZoomアプリのScopesに cloud_recording:read:list_user_recordings:admin が含まれているか確認してください。")
    raise SystemExit(1)

if not recordings:
    print(f"  ⚠️  指定された期間（{start_date} ～ {end_date}）に録画が見つかりませんでした。")
    print("      日付範囲や、クラウド録画の設定を確認してください。")
    raise SystemExit(0)

print(f"  ✅ {len(recordings)} 件の録画が見つかりました")

# ---- 転送処理 ----
print()
print("=" * 60)
print(f"転送開始：合計 {len(recordings)} 件")
print("=" * 60)
print()

success_count = 0
error_count = 0

for i, rec in enumerate(recordings, 1):
    topic = rec["topic"]
    date_str = rec["start_time"][:10] if rec["start_time"] else "日付不明"
    video_title = f"{title_prefix} {topic} ({date_str})".strip() if title_prefix else f"{topic} ({date_str})"
    description = f"Zoom録画アーカイブ\nトピック: {topic}\n録画日: {date_str}"
    tmp_path = f"/tmp/zoom_recording_{i}.mp4"

    print(f"[{i}/{len(recordings)}] {video_title}")

    try:
        download_recording(rec["download_url"], zoom_token, tmp_path)
        print("  ✅ ダウンロード完了")

        video_id = upload_to_youtube(youtube, tmp_path, video_title, description, privacy_status)
        print(f"  ✅ アップロード完了 → https://youtu.be/{video_id}")
        success_count += 1

    except Exception as e:
        error_str = str(e)
        print(f"  ❌ エラーが発生しました: {error_str}")
        print(f"  ⚠️  {japanese_error_hint(error_str)}")
        error_count += 1

    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
    print()

# ---- 完了サマリー ----
print("=" * 60)
print("転送完了！")
print(f"  ✅ 成功: {success_count} 件")
if error_count > 0:
    print(f"  ❌ 失敗: {error_count} 件")
    print()
    print("  失敗した動画は上記のエラーメッセージをご確認ください。")
    print("  YouTubeのAPI上限に達した場合は、明日以降に再度実行してください。")
print("=" * 60)
